# 02 · Distillation Experiments — the gap-closure ladder, the arms, and the program synthesis

This notebook runs the pre-registered experiment: train the compact student on three data arms
(**musdb_only** = the reused shared cell, **mixed** = MUSDB + FMA-pseudo, **distill_only** = FMA
only) with the loss and recipe held fixed, then read the **headline gap-closure ladder**
($s_{\text{base}}$, $s_{\text{mix}}$, $s_T$, $\hat C$ with CI), the distill-only and $p_{\text{FMA}}$
readings, the `mixed_trim` cross-feed, the SLR battery, the single test pass with paired stats, the
§12 interpretation branches, and the **program-level synthesis** (how D02's scaling verdict, D06's
corruption chart, and this result compose). Training/test cells are **RUN LATER** (GPU); every CPU
cell renders tested `singnet` logic.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §2 (pre-registration, recapped **verbatim**
  below), §4 (run matrix), §5 (protocol), §6 (budget), §12 (interpretation matrix);
  [`../THEORY.md`](../THEORY.md) §4 (the $\hat C$ estimand + its CI), §2 (the D06 bridge), §6 (stats).
- **Data prep + teacher labeling** are walked in
  [`01_teacher_and_data.ipynb`](01_teacher_and_data.ipynb); MUSDB prep is reused from Direction 01.
- **Un-run by design:** the launch / test cells are ⚠️ RUN-LATER banners with §6 runtimes; the
  analysis cells render on schematic/illustrative inputs (clearly labeled) so the logic is visible
  un-run, and read the real CSVs RUN LATER.

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU. Installs the pinned env and mounts Drive.
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt          # + `pip install demucs` for the teacher (RUN LATER)
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"]  = "/content/drive/MyDrive/musdb_shards"
# os.environ["PSEUDO_ROOT"] = "/content/drive/MyDrive/fma_pseudo"   # SEPARATE root (§3.3 guard)
import sys
from pathlib import Path
# scripts/ holds the committed, unit-tested pipeline (prepare_fma, teacher_label); add it to the
# path exactly as tests/ do, so the notebook IMPORTS that tested logic instead of redefining it.
sys.path.insert(0, str(Path.cwd() / "scripts"))
print("Bootstrap cell — run on Colab only (see comments). No-op here.")

## 1 · Pre-registration recap (MASTER_PLAN §2, **verbatim**)

> Notation: on the MUSDB **test** set (one pass, end of study), let
> $s_{\text{base}}$ = mean vocals SI-SDR of `musdb_only` (3 seeds),
> $s_{\text{mix}}$ = mean of `mixed` (3 seeds),
> $s_{\text{T}}$ = the teacher's own score (evaluated in the same session, same metric).
> Gap $G = s_{\text{T}} - s_{\text{base}}$; closure
> $\hat C = (s_{\text{mix}} - s_{\text{base}}) / G$. σ_seed = pooled between-seed std of
> the two 3-seed cells (validation-based decisions use the val analogue).
>
> **H-10 (gap closure).**
> - **Supported** iff $s_{\text{mix}} - s_{\text{base}} > \sigma_{\text{seed}}$ (the
>   improvement is real) **and** $\hat C \ge 0.25$ (it is big enough to matter).
> - **Partially supported** iff the improvement is real but $\hat C < 0.25$ — pseudo-
>   labels help, less than hoped; the measured $\hat C$ with CI is the finding.
> - **Refuted (null)** iff $|s_{\text{mix}} - s_{\text{base}}| \le \sigma_{\text{seed}}$.
> - **Refuted (negative transfer)** iff $s_{\text{mix}} < s_{\text{base}} - \sigma_{\text{seed}}$
>   — teacher errors/domain shift poison the student (the Direction-06 bridge branch).
> Precondition (sanity, not hypothesis): $G > 2$ dB on the frozen protocol — else the
> premise ("a large gap exists") failed upstream and the study reframes to documenting
> that (pre-registered).

**Pre-registered secondaries (descriptive, pre-written readings):** `distill_only` vs
`musdb_only` (1 seed); `mixed25` (p_FMA = 0.25, 1 seed) mixing-ratio sensitivity; `mixed_trim`
(1 seed, the Direction-06 trimmed-loss defense probe); **SLR** for every arm; and the Direction-02
scaling-verdict cross-reference that pre-frames whether "more data" was ever the right lever.

## 2 · Run matrix + shared-cell accounting (MASTER_PLAN §4.1)

Six new GPU runs; the **musdb_only** arm is the shared Direction-01 `l1mag` cell (reused, not
retrained — its config hash equals D01's, asserted below: the **sixth** reuse of `a97d5400e994`).
The cell prints the matrix and each config's data_source/p_fma — the same information
`run_sweep.py --direction 10 --dry-run` shows, inline.

In [ ]:
# CPU-runnable now: the run matrix, the shared-cell hash equality, and each config's source.
from singnet.utils.config import resolve_config, hash_config, pseudo_data_spec

D10 = "10-demucs-distillation/configs"
D01 = "01-loss-function-study/configs/l1mag_seed0_reduced.yaml"
shared = hash_config(resolve_config(D01))
base = hash_config(resolve_config(f"{D10}/base.yaml"))
print(f"musdb_only arm == D01 l1mag shared cell:  {base == shared} (= {shared}; 0 new runs)\n")

matrix = ["mixed_seed0", "mixed_seed1", "mixed_seed2",
          "distill_only_seed0", "mixed25_seed0", "mixed_trim_seed0"]
for name in matrix:
    cfg = resolve_config(f"{D10}/{name}.yaml")
    spec = pseudo_data_spec(cfg)
    print(f"  {name:22s} data_source={spec['data_source']:8s} p_fma={spec['p_fma']:.2f}  {hash_config(cfg)}")

In [ ]:
# ⚠️ RUN THIS LATER (GPU) — the prep + teacher labeling + the 6 runs.
# Prereqs: D01 MUSDB prep done; FMA labeled (notebook 01, §4). Each run ~1.3–1.8 h at REDUCED
# (16k steps); 6 runs ~8–11 T4-h; the whole direction ceiling ≈ 12–17 T4-h (MASTER_PLAN §6).
#
#   # 0. dry-run (CPU): data_source / p_fma / n_pseudo / MUSDB-path guard per config:
#   !python scripts/run_sweep.py --direction 10 --dry-run
#   # 1. the 6 runs (GPU); fully resumable (a Colab disconnect costs minutes):
#   !python scripts/run_sweep.py --direction 10 --stage reduced
print("RUN-LATER banner — the 6 GPU runs (MASTER_PLAN §6). No-op here.")

## 3 · Headline — the gap-closure ladder (MASTER_PLAN §2; THEORY §4)

The pre-registered headline: a **ladder** from $s_{\text{base}}$ (the 86-song student) up through
$s_{\text{mix}}$ (student + teacher-labeled FMA) to $s_T$ (the teacher ceiling), with the closed
fraction $\hat C = (s_{\text{mix}} - s_{\text{base}})/(s_T - s_{\text{base}})$ annotated and its CI.
The cell draws the ladder with a **schematic** layout (clearly labeled) so the figure logic is
complete un-run; **RUN LATER** it reads the real per-seed test scores.

In [ ]:
# ⚠️ RUN THIS LATER for real numbers — the gap-closure ladder (the headline, §2).
# The values below are SCHEMATIC (illustrative geometry, NOT results) so the figure code is visible
# un-run. RUN LATER: replace with the 3-seed test means (musdb_only, mixed) + the teacher s_T.
import numpy as np
import matplotlib.pyplot as plt

s_base, s_mix, s_T = 5.0, 6.4, 10.0          # SCHEMATIC dB (illustrative only)
C_hat = (s_mix - s_base) / (s_T - s_base)     # the closed fraction of the gap

fig, ax = plt.subplots(figsize=(6.2, 3.6))
for y, label, color in [(s_base, "musdb_only  $s_{base}$", "#888"),
                        (s_mix, "mixed  $s_{mix}$", "#1f77b4"),
                        (s_T, "teacher  $s_T$ (ceiling)", "#d62728")]:
    ax.axhline(y, color=color, lw=2); ax.text(1.02, y, label, va="center", fontsize=9, color=color)
ax.annotate("", xy=(0.5, s_mix), xytext=(0.5, s_base), arrowprops=dict(arrowstyle="->", color="#1f77b4"))
ax.text(0.53, (s_base + s_mix) / 2, f"$\\hat C$ ≈ {C_hat:.0%} of gap", fontsize=9, color="#1f77b4")
ax.annotate("", xy=(0.2, s_T), xytext=(0.2, s_base), arrowprops=dict(arrowstyle="<->", color="#888"))
ax.text(0.06, (s_base + s_T) / 2, "gap $G$", fontsize=9, color="#555", rotation=90, va="center")
ax.set_xlim(0, 2); ax.set_ylim(s_base - 1, s_T + 1); ax.set_xticks([])
ax.set_ylabel("vocals SI-SDR (dB)"); ax.set_title("SCHEMATIC gap-closure ladder (RUN LATER → real)")
fig.tight_layout(); plt.show()
print(f"H-10 supported iff (s_mix − s_base) > σ_seed AND Ĉ ≥ 0.25 — here Ĉ(schematic) = {C_hat:.2f}")

### 3.1 · The $\hat C$ estimand + its delta-method CI (THEORY §4)

$\hat C$ reuses Direction 06's tested `recovery_fraction` verbatim under the map
(clean → $s_T$, bleed → $s_{\text{base}}$, trim → $s_{\text{mix}}$): the linearization is identical.
The cell computes $\hat C$ and its CI on **illustrative** numbers (not results) so the estimator is
concrete and CPU-runnable; the "$d \le 0$ ⇒ not evaluable" guard **is** the $G > 2$ dB precondition.

In [ ]:
# CPU-runnable now: Ĉ + its delta-method CI via the D06 estimator (illustrative inputs).
from singnet.analysis import recovery_fraction

# map (clean=s_T, bleed=s_base, trim=s_mix); σ are the between-seed stds of the two 3-seed cells.
res = recovery_fraction(s_clean=10.0, s_bleed=5.0, s_trim=6.4,
                        sigma_clean=0.0, sigma_bleed=0.35, sigma_trim=0.40, n=3)
print(f"Ĉ = {res.rho:.3f}   95% CI [{res.ci_lo:.3f}, {res.ci_hi:.3f}]   (illustrative, not results)")
print(f"gap G = {res.degradation_db:.2f} dB (> 2 dB precondition met); evaluable = {res.evaluable}")
# the code guard: a degenerate gap (s_T ≤ s_base, d ≤ 0) returns not-evaluable rather than a
# fabricated number; the STRICTER G > 2 dB precondition (§6) is applied by the analyst on top.
degenerate = recovery_fraction(s_clean=5.0, s_bleed=5.2, s_trim=5.1)   # d = s_T − s_base ≤ 0
print(f"degenerate gap (s_T ≤ s_base) -> Ĉ evaluable? {degenerate.evaluable} (code guard: d ≤ 0)")
print("precondition: report Ĉ only when G > 2 dB (else reframe to 'why the gap is small', §6).")

## 4 · `distill_only` reading (secondary) — can pseudo-data substitute for real labels?

`distill_only` (FMA-pseudo only, 1 seed) vs `musdb_only`: can ~6.7 h of teacher-labeled CC audio
stand in for 86 real songs end to end? A provocative headline if `distill_only ≥ musdb_only`
(flagged for replication — 1 seed); grounds the "labels matter" story with a number if `≪`.
**RUN LATER** (reads the test CSV).

In [ ]:
# ⚠️ RUN THIS LATER — distill_only vs musdb_only on the MUSDB test set (1 seed; §12 branch).
#   import pandas as pd
#   df = pd.read_csv("10-demucs-distillation/results/test_session.csv")
#   d = df.set_index("arm")["sisdr_vocals_mean"]
#   print("distill_only − musdb_only =", d["distill_only"] - d["musdb_only"], "dB (flag: 1 seed)")
print("RUN-LATER banner — distill_only reading (MASTER_PLAN §12). No-op here.")

## 5 · $p_{\text{FMA}}$ sensitivity — `mixed25` vs `mixed` (secondary)

`mixed25` (p_FMA = 0.25) vs `mixed` (0.5): does the mixing ratio matter? `p_fma` is a config-hash
identity field, so the two are provably distinct runs (asserted in §2). **RUN LATER** (reads the
test CSVs).

In [ ]:
# ⚠️ RUN THIS LATER — the p_FMA axis (mixed25 @0.25 vs mixed @0.5; §12 partial branch follow-up).
#   for arm, p in [("musdb_only", 0.0), ("mixed25", 0.25), ("mixed", 0.5), ("distill_only", 1.0)]:
#       ...  # plot mean test SI-SDR vs p_FMA — the ratio sensitivity curve
print("RUN-LATER banner — p_FMA sensitivity (MASTER_PLAN §4.1). No-op here.")

## 6 · `mixed_trim` cross-feed — the Direction-06 bridge telemetry (THEORY §2)

`mixed_trim` adds Direction 06's trimmed loss (q = 0.30) as the teacher-error defense. THEORY §2:
if teacher error behaves like bleed, the trimmer's **dropped** set is **FMA-enriched** (the
corrupted targets carry the higher irreducible loss) — the falsifiable telemetry signature. If the
dropped set is pool-agnostic, trimming buys nothing (`mixed_trim ≈ mixed`) — the honest null.
**RUN LATER** (reads the trim telemetry + the pool tag).

In [ ]:
# ⚠️ RUN THIS LATER — the trim cross-feed telemetry (the D06 bridge; THEORY §2.4).
#   import pandas as pd
#   trim = pd.read_csv("checkpoints/<mixed_trim hash>/trim_energy_stats.csv")
#   # RUN LATER: join kept/dropped indices with the batch pool tag (MixedPools.pool_is_fma) and test
#   #   FMA-fraction(dropped) > FMA-fraction(kept)  -> teacher error behaves like bleed (THEORY §2.4)
#   # and read whether mixed_trim − mixed > σ_seed (does the cheap defense help?).
print("RUN-LATER banner — mixed_trim telemetry (the D06 bridge). No-op here.")

## 7 · SLR battery — does pseudo-training change silence behavior? (Direction 08)

Direction 08's Silence-Leakage Ratio (θ ∈ {−50, −60, −70}) for **every** arm: teacher leakage in
quiet passages is a plausible transfer, and karaoke products care. If SLR degrades under
pseudo-training, D08's metric earns its keep and pseudo-data would be filtered by teacher SLR
(named follow-up). **RUN LATER** (`evaluate.py --slr` writes all three θ).

In [ ]:
# ⚠️ RUN THIS LATER — the SLR battery per arm (Direction 08's metric joins the eval battery).
#   import pandas as pd
#   df = pd.read_csv("10-demucs-distillation/results/test_session.csv")
#   df[["arm", "slr_m50", "slr_m60", "slr_m70"]]   # lower is better; do-nothing anchor = 0 dB
print("RUN-LATER banner — SLR battery at θ ∈ {−50,−60,−70} (MASTER_PLAN §5). No-op here.")

## 8 · Test session + paired statistics (MASTER_PLAN §5)

Exactly one test pass: the 6 seed-cell + 3 single-seed student checkpoints + **the teacher** + the
do-nothing / oracle-IRM anchors on the 50 MUSDB test tracks, scoring vocals/accomp SI-SDR, SI-SDRi,
**SLR at all three θ**, and the museval SDR secondary. Confirmatory inference is a **paired
bootstrap + Wilcoxon** on the one pre-registered pair (`mixed` − `musdb_only`); $\hat C$'s CI is the
paired-track bootstrap of §3.1. **RUN LATER** (GPU ~1–1.5 h).

In [ ]:
# ⚠️ RUN THIS LATER (GPU ~1–1.5 h) — the single test session + paired stats.
#   # 1. score the 9 student checkpoints + teacher + anchors on the 50 test tracks, SLR at 3 θ:
#   !python scripts/evaluate.py --direction 10 --test-session --include-teacher --slr \
#       --shard-root $SHARD_ROOT --splits-csv 01-loss-function-study/configs/splits.csv \
#       --output-dir 10-demucs-distillation/results
#   # 2. the ONE pre-registered pair (mixed − musdb_only): paired bootstrap 95% CI + Wilcoxon;
#   #    σ_seed from singnet.analysis.pooled_seed_sigma over the two 3-seed cells (THEORY §6).
print("RUN-LATER banner — the single test session (MASTER_PLAN §5). No-op here.")

## 9 · Interpretation branches (MASTER_PLAN §12 — labeled stubs, filled at analysis freeze)

The verdict is pre-committed to the outcome, so no post-hoc story is possible. Each branch is a
**stub** to be completed with the frozen numbers; the full prose lives in `paper/PAPER.md`.

- **9.1 H-10 supported ($\hat C \ge 25\%$)** — pseudo-labeling closes a real gap at hobbyist
  compute; StemCraft's from-scratch engine gains a cheap upgrade path that scales with more CC
  audio (named future work). _[fill: $\hat C$ + CI, the ladder]_
- **9.2 Partial ($>σ$, $\hat C < 25\%$)** — help is real but modest, likely domain-shift-limited;
  the $p_{\text{FMA}}$ and data-volume axes become the named follow-ups. _[fill: $\hat C$ + CI]_
- **9.3 Refuted (null)** — 6.7 h of shifted pseudo-data adds nothing the 86 songs + remix didn't;
  connect to D02's verdict (§10). _[fill: $|s_{\text{mix}}-s_{\text{base}}|$ vs σ_seed]_
- **9.4 Negative transfer** — teacher errors poison the student (corrupted-target training in the
  wild); the D06 bridge quantifies and `mixed_trim` says whether the cheap defense helps. _[fill]_
- **9.5 `distill_only` ≥ / ≪ `musdb_only`** — pseudo-labels rival / cannot replace real labels
  (1 seed — flagged for replication). _[fill]_
- **9.6 SLR degrades under pseudo-training** — teacher silence-leakage transfers; karaoke products
  filter pseudo-data by teacher SLR (named follow-up). _[fill: ΔSLR per arm]_

## 10 · Conclusions + the program-level synthesis

The closing move composes **three** directions into the project's data-vs-capacity story:
**D02's scaling verdict** (was the student data-starved or saturated?), **D06's corruption chart**
(what does target corruption cost, and can trimming recover it?), and **this** result ($\hat C$).
The cell below renders that synthesis on **illustrative** inputs using the real `singnet.analysis`
functions (clearly labeled) — the logic is visible un-run and reads the frozen numbers RUN LATER.

In [ ]:
# CPU-runnable now: the program-level synthesis — D02 scaling + D06 corruption + this Ĉ.
# Inputs are ILLUSTRATIVE (not results); the composition logic is what ships.
from singnet.analysis import fit_log2, pooled_seed_sigma, recovery_fraction

# (a) D02 scaling verdict: is the student data-starved (steep log2 slope b) or saturated (flat)?
ns      = [21, 43, 64, 86]                       # subset sizes (D02 axis)
sisdr   = [3.1, 4.0, 4.6, 5.0]                   # SCHEMATIC mean val SI-SDR
fit = fit_log2(ns, sisdr)                         # score ≈ a + b·log2(n); b is dB/doubling
verdict = "data-starved" if fit.b > 0.5 else "saturated"
print(f"[D02]  log2 scaling slope b ≈ {fit.b:.2f} dB/doubling -> {verdict}")

# (b) D06 corruption chart: the price of teacher error as bleed, and trimming's recovery.
rec = recovery_fraction(s_clean=5.0, s_bleed=3.8, s_trim=4.4)   # SCHEMATIC (clean/bleed/trim dB)
print(f"[D06]  bleed cost {rec.degradation_db:.1f} dB; trimming recovers rho ≈ {rec.rho:.0%}")

# (c) THIS result: the gap the teacher-labeled data closes.
C = recovery_fraction(s_clean=10.0, s_bleed=5.0, s_trim=6.4)    # (clean=s_T, bleed=s_base, trim=s_mix)
sig = pooled_seed_sigma([5.0, 5.2, 4.8], [6.4, 6.6, 6.2])
print(f"[D10]  Ĉ ≈ {C.rho:.0%} of the student->teacher gap; sigma_seed ≈ {sig:.2f} dB")

print()
print("SYNTHESIS (fill at freeze): if D02 says data-starved, the KIND of data (teacher-labeled,")
print("  domain-shifted) is the lever this direction tests; D06 prices the teacher-error cost the")
print("  Ĉ pays; together they place pseudo-labeling on the project's data-vs-capacity map.")

### The one-paragraph takeaway (filled at analysis freeze)

_Whether the compact student can close a meaningful fraction of its gap to the shipped teacher from
license-safe public audio at a GPU weekend — the $\hat C$ with its CI — decides it, read against
D02's scaling verdict (was more data ever the lever?) and D06's corruption chart (what did the
teacher's errors cost?). The full prose, with the frozen ladder and the paired test, lives in
`paper/PAPER.md`._